# Montreal Forced Aligner (MFA)

A refresher on **Montreal Forced Aligner** — the de-facto open-source tool for *forced
alignment*: given an audio recording **and** the transcript of what was said, MFA finds the
start/end time of every word and every phone and writes them to a Praat **TextGrid**.

**Domain:** Data Analysis & Research  ·  **runnable:** yes (the alignment engine is a conda
CLI; the Python cells below build a real corpus on disk and parse real MFA output).

## 1. What & Why

**What it is.** MFA solves *forced alignment*. You already know **what** was said (you have the
transcript); you need to know **where** each unit lands on the audio timeline. Feed it
`audio + transcript + a pronunciation dictionary + an acoustic model` and it returns precise
word- and phone-level time boundaries. It is built on **Kaldi** (GMM-HMM acoustic models) and
ships pretrained dictionaries and acoustic models for ~30 languages.

**The problem it solves.** Hand-aligning speech in Praat is brutal — minutes of audio take hours
to segment by ear. MFA does it in seconds per utterance, reproducibly, at phone granularity.

**When to reach for it.**
- Phonetics / sociolinguistics research (measure vowel durations, formants at the right instant).
- **TTS / voice-cloning dataset prep** — most neural TTS pipelines want phone-level alignments.
- Lip-sync, subtitle timing, audio-book chaptering, pronunciation-training apps.
- Any time you have *clean audio + a known transcript* and need timestamps.

**When NOT to.** It is **not** speech recognition — it cannot tell you *what* was said, only
*where*. If you don't have a transcript, run ASR (Whisper) first. For sentence-level timing only,
lighter tools (`aeneas`) are simpler. For noisy/spontaneous speech, expect to clean OOVs and tune
beams.

## 2. Mental Model

> **You already have the script of the movie; MFA just figures out the timecodes.**

Picture the transcript expanded into its phone sequence (via the pronunciation dictionary):
`the quick fox` → `DH AH | K W IH K | F AA K S`. The acoustic model is a Hidden Markov Model
whose states emit those phones. MFA runs **Viterbi** to find the single most probable path that
threads that *fixed, known* phone sequence onto the audio frames (10 ms each). Because the
sequence is fixed, the only unknown is the **timing of the transitions** — so the search is far
easier and far more accurate than open-vocabulary recognition.

```
transcript ──(dictionary)──▶ phone sequence ──┐
                                              ├─▶ Viterbi over audio frames ─▶ time boundaries
audio ──(MFCC features, 10ms)─────────────────┘                              (words + phones)
```

Two refinement passes matter: a first **monophone/triphone** alignment, then **speaker
adaptation (fMLLR)** that warps the model to each speaker's voice and re-aligns — which is why
MFA aligns *per speaker* and wants speaker labels.

## 3. Key Concepts

- **Corpus.** A directory of paired files: `utt.wav` + `utt.lab` (or `.txt`) holding that clip's
  orthographic transcript. Sub-folders are treated as **speakers** (used for adaptation). Audio
  should be mono; MFA resamples to 16 kHz internally.
- **Pronunciation dictionary.** Maps each orthographic word to one or more phone sequences
  (`english_us_arpa`, IPA dicts, etc.). The phone set here **must** match the acoustic model's.
- **Acoustic model.** Pretrained GMM-HMM (e.g. `english_us_arpa`) downloaded with
  `mfa model download`. Maps audio features → phone-state likelihoods.
- **G2P model.** Grapheme-to-phoneme model that *generates* pronunciations for words missing from
  the dictionary (**OOV** words) — run `mfa g2p` to fill gaps before aligning.
- **TextGrid.** The output, a Praat annotation file with **interval tiers** — typically a `words`
  tier and a `phones` tier, each interval carrying `xmin`, `xmax`, `text`.
- **OOV (out-of-vocabulary).** A transcript word with no dictionary entry — by default it is
  treated as unknown and its segment is unreliable. `mfa validate` reports OOVs first.
- **Beam / retry-beam.** Viterbi search width. Utterances that fail at the default beam are
  retried at `retry_beam`; chronic failures usually mean a transcript/audio mismatch.
- **Speaker adaptation (fMLLR).** The second pass that fits the model to each speaker; the reason
  alignments improve when speaker folders are correct.

## 4. Setup

MFA is distributed through **conda-forge** (it pulls in Kaldi + OpenFst binaries — there is no
pure-`pip` install):

```bash
conda create -n aligner -c conda-forge montreal-forced-aligner
conda activate aligner

# one-time: fetch a pretrained dictionary + acoustic model
mfa model download dictionary english_us_arpa
mfa model download acoustic  english_us_arpa
```

A typical run is three CLI steps — **validate, then align**:

```bash
mfa validate ./corpus english_us_arpa english_us_arpa          # catch OOVs / format issues
mfa align    ./corpus english_us_arpa english_us_arpa ./out    # writes ./out/<speaker>/<utt>.TextGrid
```

The Python cells below **don't require MFA to be installed** — they build the corpus layout MFA
expects and parse the TextGrid format it emits, so the notebook runs end-to-end in a fresh
kernel. The one cell that shells out to `mfa` is gated behind a `shutil.which` check.

In [1]:
import shutil, sys

mfa = shutil.which("mfa")
print("Python      :", sys.version.split()[0])
print("MFA CLI     :", mfa or "not installed  →  conda install -c conda-forge montreal-forced-aligner")
print("Std-lib only:", "wave, wave-file + TextGrid parsing below need no third-party packages")

Python      : 3.13.7
MFA CLI     : not installed  →  conda install -c conda-forge montreal-forced-aligner
Std-lib only: wave, wave-file + TextGrid parsing below need no third-party packages


## 5. Worked Examples

Three examples: **(1)** build the on-disk corpus MFA expects, **(2)** parse a TextGrid exactly like
the one MFA writes and compute phone durations, **(3)** the real `mfa align` invocation, gated so
it only runs if the CLI is present.

### Example 1 — Build the corpus layout MFA expects

MFA reads a directory of `<audio>` + `<transcript>` pairs, with sub-folders as speakers. Here we
create one speaker, a real 16 kHz mono WAV (silent — we only need valid headers), and its `.lab`
transcript, then print the tree.

In [2]:
import pathlib, tempfile, wave

corpus = pathlib.Path(tempfile.mkdtemp(prefix="mfa_corpus_"))
spk = corpus / "speaker1"
spk.mkdir()

# a valid 0.5 s, 16 kHz, mono, 16-bit WAV (silent frames — headers are what matter here)
with wave.open(str(spk / "utt1.wav"), "w") as w:
    w.setnchannels(1)        # mono
    w.setsampwidth(2)        # 16-bit
    w.setframerate(16000)    # 16 kHz, MFA's working rate
    w.writeframes(b"\x00\x00" * 8000)   # 8000 frames = 0.5 s

# paired transcript: one .lab (or .txt) per audio file, same stem
(spk / "utt1.lab").write_text("the quick brown fox")

print("corpus root:", corpus)
for p in sorted(corpus.rglob("*")):
    kind = "dir " if p.is_dir() else f"{p.stat().st_size:>5} B"
    print(f"  {kind}  {p.relative_to(corpus)}")

corpus root: /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/mfa_corpus_5s37mnkj
  dir   speaker1
     19 B  speaker1/utt1.lab
  16044 B  speaker1/utt1.wav


### Example 2 — Parse a TextGrid and measure phone durations

This is a trimmed **long-format Praat TextGrid** in exactly the shape MFA writes to the output
directory: a `words` tier and a `phones` tier of timed intervals. We parse it with the standard
library and compute per-phone durations — the bread-and-butter of phonetic analysis.

In [3]:
import re

textgrid = '''File type = "ooTextFile"
Object class = "TextGrid"

xmin = 0
xmax = 0.9
tiers? <exists>
size = 2
item []:
    item [1]:
        class = "IntervalTier"
        name = "words"
        xmin = 0
        xmax = 0.9
        intervals: size = 2
        intervals [1]:
            xmin = 0
            xmax = 0.45
            text = "the"
        intervals [2]:
            xmin = 0.45
            xmax = 0.9
            text = "quick"
    item [2]:
        class = "IntervalTier"
        name = "phones"
        xmin = 0
        xmax = 0.9
        intervals: size = 6
        intervals [1]:
            xmin = 0
            xmax = 0.18
            text = "DH"
        intervals [2]:
            xmin = 0.18
            xmax = 0.45
            text = "AH"
        intervals [3]:
            xmin = 0.45
            xmax = 0.58
            text = "K"
        intervals [4]:
            xmin = 0.58
            xmax = 0.70
            text = "W"
        intervals [5]:
            xmin = 0.70
            xmax = 0.80
            text = "IH"
        intervals [6]:
            xmin = 0.80
            xmax = 0.90
            text = "K"
'''

def parse_textgrid(tg):
    """Return {tier_name: [(xmin, xmax, text), ...]} from a long-format TextGrid."""
    tiers, current = {}, None
    name_re = re.compile(r'name = "([^"]*)"')
    iv_re   = re.compile(r'xmin = ([\d.]+)\s+xmax = ([\d.]+)\s+text = "([^"]*)"')
    # split into tier blocks on each `item [n]:`
    for block in re.split(r'item \[\d+\]:', tg)[1:]:
        m = name_re.search(block)
        if not m:
            continue
        intervals = [(float(a), float(b), t) for a, b, t in iv_re.findall(block)]
        tiers[m.group(1)] = intervals
    return tiers

tiers = parse_textgrid(textgrid)
print("tiers:", list(tiers))
print()
print(f"{'phone':>6}  {'start':>5}  {'end':>5}  {'dur (ms)':>8}")
for xmin, xmax, phone in tiers["phones"]:
    if phone:  # skip empty (silence) intervals
        print(f"{phone:>6}  {xmin:5.2f}  {xmax:5.2f}  {(xmax - xmin) * 1000:8.0f}")

tiers: ['words', 'phones']

 phone  start    end  dur (ms)
    DH   0.00   0.18       180
    AH   0.18   0.45       270
     K   0.45   0.58       130
     W   0.58   0.70       120
    IH   0.70   0.80       100
     K   0.80   0.90       100


### Example 3 — The real alignment call (gated on the CLI)

This is the actual command MFA runs. It's wrapped in a `shutil.which("mfa")` guard so the
notebook still executes top-to-bottom when MFA isn't installed — you always see the exact
invocation, and it really runs when the CLI is present.

In [4]:
import shutil, subprocess

dictionary, acoustic = "english_us_arpa", "english_us_arpa"
out_dir = corpus.parent / (corpus.name + "_aligned")

cmd = ["mfa", "align", str(corpus), dictionary, acoustic, str(out_dir), "--clean"]
print("Would run:\n  " + " ".join(cmd))

if shutil.which("mfa"):
    # validate first in practice: mfa validate <corpus> <dict> <acoustic>
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print("\nreturn code:", proc.returncode)
    print(proc.stdout[-800:] or proc.stderr[-800:])
else:
    print("\nmfa CLI not found — skipping the real run "
          "(install via conda-forge and download the english_us_arpa models to execute).")

Would run:
  mfa align /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/mfa_corpus_5s37mnkj english_us_arpa english_us_arpa /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/mfa_corpus_5s37mnkj_aligned --clean

mfa CLI not found — skipping the real run (install via conda-forge and download the english_us_arpa models to execute).


## 6. Gotchas & Pitfalls

- **Forced alignment is not recognition.** MFA *requires* a correct transcript. Wrong/garbled
  transcripts don't error loudly — they silently produce bad boundaries.
- **OOV words.** Any word missing from the dictionary is unaligned/unreliable. Always
  `mfa validate` first, then close gaps with `mfa g2p` (generate pronunciations) or hand-add
  entries. Numbers, names, and slang are the usual culprits.
- **Phone set mismatch.** The dictionary and the acoustic model **must** share a phone set
  (e.g. both `english_us_arpa`). Mixing an IPA dict with an ARPA model fails or aligns garbage.
- **Audio format.** Use mono; MFA resamples to 16 kHz. Stereo, odd sample rates, or
  variable-bit-rate MP3s cause subtle problems — convert to 16 kHz mono WAV first.
- **Speaker folders matter.** Speaker adaptation (fMLLR) is per-speaker; dumping every clip into
  one folder (or one clip per folder) weakens adaptation. Group a speaker's utterances together.
- **Beam errors.** `Utterance ... failed to align` at the default beam usually means a
  transcript/audio mismatch or very noisy audio, not a beam that's too small — fix the transcript
  before cranking `--beam`/`--retry_beam`.
- **Conda only.** There's no reliable `pip install` — it depends on compiled Kaldi/OpenFst. Use
  the conda-forge package or the Docker image.
- **Long unsegmented files.** MFA aligns utterance-by-utterance; multi-minute files should be
  pre-segmented (or use `mfa align` with VAD-based segmentation) rather than aligned whole.

## 7. When to Use vs Alternatives

| Tool | Granularity | Strengths | Weaknesses vs MFA |
|------|-------------|-----------|-------------------|
| **MFA** | word **+ phone** | Pretrained models for ~30 langs, speaker adaptation, accurate phone boundaries, reproducible CLI | Conda-only, needs a dictionary + transcript, steeper setup |
| **WhisperX / wav2vec2 CTC** | word (phone via CTC segmentation) | No dictionary needed, pip-installable, GPU-fast, pairs with ASR so you don't need a transcript | Phone-level alignment is less precise; large models/GPU |
| **aeneas** | sentence / fragment | Dead simple, great for audiobook ↔ text sync | No phone alignment; coarse |
| **Gentle** | word + phone | Web-UI, Kaldi-based, easy to try | English-only, less maintained, fewer models |
| **Praat (manual / TextGrid)** | anything | Total control, the analysis tool everyone uses downstream | Hand-segmenting is hours of work; not scalable |

**Rule of thumb:** you have a transcript and need **accurate phone boundaries across languages** →
**MFA**. You *don't* have a transcript, or only need word timings → **WhisperX**. You only need
sentence/fragment sync → **aeneas**.

## 8. Resources

- **Official docs** — installation, the `align`/`validate`/`g2p` commands, corpus format:
  <https://montreal-forced-aligner.readthedocs.io/>
- **Source on GitHub** — issues, releases, examples:
  <https://github.com/MontrealCorpusTools/Montreal-Forced-Aligner>
- **Pretrained models index** — dictionaries, acoustic, and G2P models per language:
  <https://mfa-models.readthedocs.io/>
- **Original paper** — McAuliffe et al., *Montreal Forced Aligner* (Interspeech 2017):
  <https://www.isca-archive.org/interspeech_2017/mcauliffe17_interspeech.html>
- **Praat TextGrid format** — the output you'll parse and visualize:
  <https://www.fon.hum.uva.nl/praat/manual/TextGrid_file_formats.html>